In [1]:
from __future__ import annotations

import copy
import json
from pathlib import Path
from typing import Any

In [2]:
RAW_CONFIG_ROOT = Path(
    "/home/blue2959/monotonic_tts/exp_configs/v2.3/conv1d_rf_sweep/raw_configs"
)

# 기준 config가 들어 있는 폴더
TEMPLATE_DIR = (
    RAW_CONFIG_ROOT
    / "conv_RF=l2_dec_RF=1_seed=1234"
)

DATA_CONFIG_TEMPLATE_PATH = TEMPLATE_DIR / "data_config.json"
MODEL_CONFIG_TEMPLATE_PATH = TEMPLATE_DIR / "model_config.json"

assert DATA_CONFIG_TEMPLATE_PATH.is_file()
assert MODEL_CONFIG_TEMPLATE_PATH.is_file()

print(f"Raw config root : {RAW_CONFIG_ROOT}")
print(f"Template        : {TEMPLATE_DIR}")

Raw config root : /home/blue2959/monotonic_tts/exp_configs/v2.3/conv1d_rf_sweep/raw_configs
Template        : /home/blue2959/monotonic_tts/exp_configs/v2.3/conv1d_rf_sweep/raw_configs/conv_RF=l2_dec_RF=1_seed=1234


In [3]:
def load_json(path: Path) -> dict[str, Any]:
    with path.open("r", encoding="utf-8") as file:
        return json.load(file)


data_config_template = load_json(
    DATA_CONFIG_TEMPLATE_PATH
)
model_config_template = load_json(
    MODEL_CONFIG_TEMPLATE_PATH
)

print("Templates loaded.")

Templates loaded.


In [4]:
SEEDS = [1234, 1235, 1236]

ALIGNER_CONDITIONS = ["l2", 1, 5, 9, 13]
DECODER_RFS = [1, 5, 9, 13, 17]


CONV_RF_CONFIGS = {
    1: {
        "conv_num_layers": 1,
        "conv_kernel_size": [1, 1],
    },
    5: {
        "conv_num_layers": 2,
        "conv_kernel_size": [3, 3],
    },
    9: {
        "conv_num_layers": 4,
        "conv_kernel_size": [3, 3],
    },
    13: {
        "conv_num_layers": 6,
        "conv_kernel_size": [3, 3],
    },
}


def decoder_rf_to_kernel_sizes(
    decoder_rf: int,
) -> list[int]:
    """
    stride=1, dilation=1인 Conv1d decoder.

    RF=1  -> [1]
    RF=5  -> [3, 3]
    RF=9  -> [3, 3, 3, 3]
    RF=13 -> [3, 3, 3, 3, 3, 3]
    RF=17 -> [3, 3, 3, 3, 3, 3, 3, 3]
    """
    if decoder_rf == 1:
        return [1]

    if decoder_rf < 1 or decoder_rf % 2 == 0:
        raise ValueError(
            "decoder_rf must be a positive odd integer, "
            + f"got {decoder_rf}"
        )

    num_layers = (decoder_rf - 1) // 2
    return [3] * num_layers


for condition in ALIGNER_CONDITIONS:
    if condition == "l2":
        print(
            "conv_RF=l2 -> unary_network_type=l2 "
            + "(Conv config ignored)"
        )
    else:
        print(
            f"conv_RF={condition:<2} -> "
            + f"{CONV_RF_CONFIGS[condition]}"
        )

print()

for decoder_rf in DECODER_RFS:
    print(
        f"decoder RF={decoder_rf:2d} -> "
        + f"kernel_sizes="
        + f"{decoder_rf_to_kernel_sizes(decoder_rf)}"
    )

conv_RF=l2 -> unary_network_type=l2 (Conv config ignored)
conv_RF=1  -> {'conv_num_layers': 1, 'conv_kernel_size': [1, 1]}
conv_RF=5  -> {'conv_num_layers': 2, 'conv_kernel_size': [3, 3]}
conv_RF=9  -> {'conv_num_layers': 4, 'conv_kernel_size': [3, 3]}
conv_RF=13 -> {'conv_num_layers': 6, 'conv_kernel_size': [3, 3]}

decoder RF= 1 -> kernel_sizes=[1]
decoder RF= 5 -> kernel_sizes=[3, 3]
decoder RF= 9 -> kernel_sizes=[3, 3, 3, 3]
decoder RF=13 -> kernel_sizes=[3, 3, 3, 3, 3, 3]
decoder RF=17 -> kernel_sizes=[3, 3, 3, 3, 3, 3, 3, 3]


In [5]:
def save_json(
    config: dict[str, Any],
    path: Path,
) -> None:
    with path.open("w", encoding="utf-8") as file:
        json.dump(
            config,
            file,
            ensure_ascii=False,
            indent=4,
        )

        # POSIX text file 마지막 newline
        file.write("\n")

In [6]:
def build_data_config(
    *,
    seed: int,
    experiment_name: str,
) -> dict[str, Any]:
    config = copy.deepcopy(data_config_template)

    config["train"]["seed"] = seed
    config["dataset"]["seed"] = seed
    config["extra_exp"]["exp_name"] = experiment_name

    return config


def build_model_config(
    *,
    aligner_condition: str | int,
    decoder_rf: int,
) -> dict[str, Any]:
    config = copy.deepcopy(model_config_template)

    aligner_config = config["nd_aligner"]["aligner"]
    decoder_config = config["nd_aligner"]["spec_dec"]

    # --------------------------------------------------------
    # Aligner unary scorer
    # --------------------------------------------------------

    if aligner_condition == "l2":
        aligner_config["unary_network_type"] = "l2"

        # conv_num_layers와 conv_kernel_size는 l2에서 무시됨.
        # 따라서 template 값을 그대로 둠.

    else:
        if aligner_condition not in CONV_RF_CONFIGS:
            raise ValueError(
                "Unsupported aligner condition: "
                + f"{aligner_condition}"
            )

        conv_config = CONV_RF_CONFIGS[
            aligner_condition
        ]

        aligner_config["unary_network_type"] = "conv"
        aligner_config["conv_num_layers"] = (
            conv_config["conv_num_layers"]
        )
        aligner_config["conv_kernel_size"] = (
            conv_config["conv_kernel_size"]
        )

    # --------------------------------------------------------
    # Conv1d decoder
    # --------------------------------------------------------

    decoder_config["decoder_type"] = "conv1d"
    decoder_config["kernel_sizes"] = (
        decoder_rf_to_kernel_sizes(decoder_rf)
    )
    decoder_config["dilation_base"] = 1

    return config

In [7]:
experiment_specs = []

for aligner_condition in ALIGNER_CONDITIONS:
    for decoder_rf in DECODER_RFS:
        for seed in SEEDS:
            experiment_name = (
                f"conv_RF={aligner_condition}"
                f"_dec_RF={decoder_rf}"
                f"_seed={seed}"
            )

            experiment_specs.append(
                {
                    "aligner_condition": aligner_condition,
                    "decoder_rf": decoder_rf,
                    "seed": seed,
                    "experiment_name": experiment_name,
                }
            )

expected_num_experiments = (
    len(ALIGNER_CONDITIONS)
    * len(DECODER_RFS)
    * len(SEEDS)
)

assert len(experiment_specs) == expected_num_experiments

print(
    f"Number of experiments: "
    + f"{len(experiment_specs)}"
)

for spec in experiment_specs:
    print(spec["experiment_name"])

Number of experiments: 75
conv_RF=l2_dec_RF=1_seed=1234
conv_RF=l2_dec_RF=1_seed=1235
conv_RF=l2_dec_RF=1_seed=1236
conv_RF=l2_dec_RF=5_seed=1234
conv_RF=l2_dec_RF=5_seed=1235
conv_RF=l2_dec_RF=5_seed=1236
conv_RF=l2_dec_RF=9_seed=1234
conv_RF=l2_dec_RF=9_seed=1235
conv_RF=l2_dec_RF=9_seed=1236
conv_RF=l2_dec_RF=13_seed=1234
conv_RF=l2_dec_RF=13_seed=1235
conv_RF=l2_dec_RF=13_seed=1236
conv_RF=l2_dec_RF=17_seed=1234
conv_RF=l2_dec_RF=17_seed=1235
conv_RF=l2_dec_RF=17_seed=1236
conv_RF=1_dec_RF=1_seed=1234
conv_RF=1_dec_RF=1_seed=1235
conv_RF=1_dec_RF=1_seed=1236
conv_RF=1_dec_RF=5_seed=1234
conv_RF=1_dec_RF=5_seed=1235
conv_RF=1_dec_RF=5_seed=1236
conv_RF=1_dec_RF=9_seed=1234
conv_RF=1_dec_RF=9_seed=1235
conv_RF=1_dec_RF=9_seed=1236
conv_RF=1_dec_RF=13_seed=1234
conv_RF=1_dec_RF=13_seed=1235
conv_RF=1_dec_RF=13_seed=1236
conv_RF=1_dec_RF=17_seed=1234
conv_RF=1_dec_RF=17_seed=1235
conv_RF=1_dec_RF=17_seed=1236
conv_RF=5_dec_RF=1_seed=1234
conv_RF=5_dec_RF=1_seed=1235
conv_RF=5_dec_RF=1_

In [8]:
created_experiments = []

for spec in experiment_specs:
    aligner_condition = spec["aligner_condition"]
    decoder_rf = spec["decoder_rf"]
    seed = spec["seed"]
    experiment_name = spec["experiment_name"]

    experiment_dir = RAW_CONFIG_ROOT / experiment_name
    experiment_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    data_config = build_data_config(
        seed=seed,
        experiment_name=experiment_name,
    )

    model_config = build_model_config(
        aligner_condition=aligner_condition,
        decoder_rf=decoder_rf,
    )

    save_json(
        data_config,
        experiment_dir / "data_config.json",
    )
    save_json(
        model_config,
        experiment_dir / "model_config.json",
    )

    created_experiments.append(experiment_dir)

print(
    f"Created/updated {len(created_experiments)} "
    + "experiment directories."
)

Created/updated 75 experiment directories.


In [9]:
for spec in experiment_specs:
    aligner_condition = spec["aligner_condition"]
    expected_decoder_rf = spec["decoder_rf"]
    expected_seed = spec["seed"]
    experiment_name = spec["experiment_name"]

    experiment_dir = RAW_CONFIG_ROOT / experiment_name

    data_config_path = experiment_dir / "data_config.json"
    model_config_path = experiment_dir / "model_config.json"

    assert data_config_path.is_file()
    assert model_config_path.is_file()

    data_config = load_json(data_config_path)
    model_config = load_json(model_config_path)

    aligner_config = model_config["nd_aligner"]["aligner"]
    decoder_config = model_config["nd_aligner"]["spec_dec"]

    # --------------------------------------------------------
    # Data config
    # --------------------------------------------------------

    assert (
        data_config["extra_exp"]["exp_name"]
        == experiment_name
    )
    assert data_config["train"]["seed"] == expected_seed
    assert data_config["dataset"]["seed"] == expected_seed

    # --------------------------------------------------------
    # Aligner config
    # --------------------------------------------------------

    if aligner_condition == "l2":
        assert (
            aligner_config["unary_network_type"]
            == "l2"
        )

    else:
        expected_conv_config = CONV_RF_CONFIGS[
            aligner_condition
        ]

        assert (
            aligner_config["unary_network_type"]
            == "conv"
        )
        assert (
            aligner_config["conv_num_layers"]
            == expected_conv_config["conv_num_layers"]
        )
        assert (
            aligner_config["conv_kernel_size"]
            == expected_conv_config["conv_kernel_size"]
        )

    # --------------------------------------------------------
    # Decoder config
    # --------------------------------------------------------

    assert decoder_config["decoder_type"] == "conv1d"
    assert (
        decoder_config["kernel_sizes"]
        == decoder_rf_to_kernel_sizes(
            expected_decoder_rf
        )
    )
    assert decoder_config["dilation_base"] == 1

print(
    f"All {len(experiment_specs)} configurations "
    + "verified successfully."
)

All 75 configurations verified successfully.
